# Data Types and Casting

## Overview
Every column in a SQL table has a declared type. Getting types right prevents silent truncation, wrong arithmetic, failed comparisons, and data pipeline bugs that are very hard to trace.

**SQLite type system — type affinity (flexible/dynamic):**

| Declared keyword | Affinity | Stored as |
|---|---|---|
| `INTEGER`, `INT`, `BIGINT` | INTEGER | 64-bit signed integer |
| `REAL`, `FLOAT`, `DOUBLE` | REAL | 8-byte IEEE 754 float |
| `TEXT`, `VARCHAR`, `CHAR` | TEXT | UTF-8 string |
| `BLOB` | BLOB | Raw bytes, no coercion |
| `NUMERIC`, `DECIMAL` | NUMERIC | Chooses integer or real |
| (anything else) | NUMERIC | Same |

SQLite will store whatever you insert — a string in an INTEGER column is legal in SQLite but raises an error in PostgreSQL. This makes SQLite convenient for prototyping but can mask bugs.

**PostgreSQL type system — strict:**
PostgreSQL enforces declared types. Inserting `'abc'` into an `INTEGER` column raises a type error immediately. Key types: `SMALLINT`, `INTEGER`, `BIGINT`, `NUMERIC(p,s)`, `REAL`, `DOUBLE PRECISION`, `TEXT`, `VARCHAR(n)`, `CHAR(n)`, `DATE`, `TIMESTAMP`, `TIMESTAMPTZ`, `BOOLEAN`, `JSONB`, `UUID`.

**Explicit casting:**
- `CAST(value AS type)` — SQL standard, works in SQLite and PostgreSQL
- `value::type` — PostgreSQL shorthand (`'2023-01-01'::DATE`)

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur  = conn.cursor()

# Lab results — result_val stored as TEXT (very common in real EHR data exports)
cur.executescript("""
CREATE TABLE lab_results (
    result_id   INTEGER PRIMARY KEY,
    patient_id  INTEGER,
    test_name   TEXT,
    result_val  TEXT,       -- numeric values stored as text strings (common in HL7/EHR exports)
    units       TEXT,
    collected   TEXT,       -- ISO date string YYYY-MM-DD
    flagged     INTEGER     -- 1 = outside reference range
);

INSERT INTO lab_results VALUES
  (1,  1,  'HbA1c',      '7.2',   '%',       '2023-03-10', 0),
  (2,  2,  'HbA1c',      '9.1',   '%',       '2023-03-15', 1),
  (3,  3,  'Creatinine', '88.5',  'umol/L',  '2023-04-01', 0),
  (4,  4,  'Creatinine', '145.0', 'umol/L',  '2023-04-12', 1),
  (5,  5,  'HbA1c',      '5.8',   '%',       '2023-05-05', 0),
  (6,  6,  'Sodium',     '138',   'mmol/L',  '2023-05-20', 0),
  (7,  7,  'Sodium',     '151',   'mmol/L',  '2023-06-01', 1),
  (8,  1,  'Creatinine', '72.3',  'umol/L',  '2023-06-18', 0),
  (9,  8,  'HbA1c',      '6.4',   '%',       '2023-07-02', 0),
  (10, 9,  'Creatinine', '210.0', 'umol/L',  '2023-07-15', 1);
""")
conn.commit()

print('lab_results table ready — result_val stored as TEXT (simulating real EHR export)')
print()
print(pd.read_sql('SELECT * FROM lab_results', conn).to_string(index=False))


lab_results table ready — result_val stored as TEXT (simulating real EHR export)

 result_id  patient_id  test_name result_val  units  collected  flagged
         1           1      HbA1c        7.2      % 2023-03-10        0
         2           2      HbA1c        9.1      % 2023-03-15        1
         3           3 Creatinine       88.5 umol/L 2023-04-01        0
         4           4 Creatinine      145.0 umol/L 2023-04-12        1
         5           5      HbA1c        5.8      % 2023-05-05        0
         6           6     Sodium        138 mmol/L 2023-05-20        0
         7           7     Sodium        151 mmol/L 2023-06-01        1
         8           1 Creatinine       72.3 umol/L 2023-06-18        0
         9           8      HbA1c        6.4      % 2023-07-02        0
        10           9 Creatinine      210.0 umol/L 2023-07-15        1


---
## SQLite type affinity and TYPEOF()

`TYPEOF()` returns the storage class of the actual stored value — not necessarily the declared column type. This reveals SQLite's dynamic typing.


In [2]:
# TYPEOF reveals actual storage class
print('=== Storage class of result_val (declared TEXT) ===')
q = """
SELECT result_id, result_val, TYPEOF(result_val) AS storage_class
FROM   lab_results
LIMIT  5
"""
print(pd.read_sql(q, conn).to_string(index=False))

# String comparison vs numeric comparison — the key danger
print()
print('=== String ORDER vs numeric ORDER (same data, different result) ===')

# This looks fine for HbA1c because '5.8' < '7.2' < '9.1' alphabetically too
# but try values like '9' vs '88':
cur = conn.cursor()
r = cur.execute("SELECT '9' > '88'").fetchone()[0]
print(f"  '9' > '88' as string comparison: {r}  (1 = TRUE)")
print(f"  9 > 88 as numeric comparison:    {0}  (0 = FALSE)")
print()
print('  String comparison goes char-by-char: chr(9) > chr(8), so 9 > 88 as strings.')
print('  This will silently mis-sort numeric data stored as text.')

print()
print('=== Sorting HbA1c values — string vs numeric ===')
q2 = """
SELECT result_val                     AS text_value,
       CAST(result_val AS REAL)       AS numeric_value
FROM   lab_results
WHERE  test_name = 'HbA1c'
ORDER BY result_val    -- string sort
"""
q3 = q2.replace('ORDER BY result_val', 'ORDER BY CAST(result_val AS REAL)')
print('String sort:'); print(pd.read_sql(q2, conn).to_string(index=False))
print(); print('Numeric sort:'); print(pd.read_sql(q3, conn).to_string(index=False))


=== Storage class of result_val (declared TEXT) ===
 result_id result_val storage_class
         1        7.2          text
         2        9.1          text
         3       88.5          text
         4      145.0          text
         5        5.8          text

=== String ORDER vs numeric ORDER (same data, different result) ===
  '9' > '88' as string comparison: 1  (1 = TRUE)
  9 > 88 as numeric comparison:    0  (0 = FALSE)

  String comparison goes char-by-char: chr(9) > chr(8), so 9 > 88 as strings.
  This will silently mis-sort numeric data stored as text.

=== Sorting HbA1c values — string vs numeric ===
String sort:
text_value  numeric_value
       5.8            5.8
       6.4            6.4
       7.2            7.2
       9.1            9.1

Numeric sort:
text_value  numeric_value
       5.8            5.8
       6.4            6.4
       7.2            7.2
       9.1            9.1


---
## CAST: explicit type conversion


In [3]:
# CAST to REAL for numeric operations
print('=== CAST result_val to REAL for arithmetic ===')
q = """
SELECT test_name,
       result_val                              AS raw_text,
       CAST(result_val AS REAL)               AS as_real,
       ROUND(CAST(result_val AS REAL), 1)     AS rounded,
       CAST(result_val AS INTEGER)            AS as_int   -- truncates decimal
FROM   lab_results
ORDER BY test_name, CAST(result_val AS REAL)
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Integer division — the classic trap
print()
print('=== Integer division vs real division ===')
q2 = """
SELECT
    7 / 2                       AS int_div,         -- 3 (both operands are integers)
    7 / 2.0                     AS real_div,        -- 3.5
    7.0 / 2                     AS real_div2,       -- 3.5
    CAST(7 AS REAL) / 2         AS cast_div,        -- 3.5
    CAST(7 AS REAL) / CAST(2 AS REAL)  AS both_cast -- 3.5
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print()
print('PostgreSQL behaves the same: 7/2 = 3, 7/2.0 = 3.5')
print('Rate calculations like COUNT(*)/total will silently return 0 if both are integers.')


=== CAST result_val to REAL for arithmetic ===
 test_name raw_text  as_real  rounded  as_int
Creatinine     72.3     72.3     72.3      72
Creatinine     88.5     88.5     88.5      88
Creatinine    145.0    145.0    145.0     145
Creatinine    210.0    210.0    210.0     210
     HbA1c      5.8      5.8      5.8       5
     HbA1c      6.4      6.4      6.4       6
     HbA1c      7.2      7.2      7.2       7
     HbA1c      9.1      9.1      9.1       9
    Sodium      138    138.0    138.0     138
    Sodium      151    151.0    151.0     151

=== Integer division vs real division ===
 int_div  real_div  real_div2  cast_div  both_cast
       3       3.5        3.5       3.5        3.5

PostgreSQL behaves the same: 7/2 = 3, 7/2.0 = 3.5
Rate calculations like COUNT(*)/total will silently return 0 if both are integers.


---
## Date strings and date type handling


In [4]:
# SQLite: dates stored as ISO TEXT — functions treat them as dates
print('=== Date functions on TEXT date columns ===')
q = """
SELECT result_id,
       collected,
       DATE(collected)                                   AS parsed_date,
       STRFTIME('%Y', collected)                         AS year,
       STRFTIME('%m', collected)                         AS month,
       STRFTIME('%Y-%m', collected)                      AS year_month,
       CAST(JULIANDAY('2023-12-31') -
            JULIANDAY(collected) AS INTEGER)             AS days_before_year_end
FROM   lab_results
ORDER BY collected
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('=== PostgreSQL date/time types (reference — not runnable in SQLite) ===')
print("""
  -- PostgreSQL uses native typed columns:
  collected DATE        -- stores as a 4-byte date, no time component
  collected TIMESTAMP   -- date + time, no timezone
  collected TIMESTAMPTZ -- date + time + timezone (recommended for events)

  -- Casting syntax:
  '2023-03-10'::DATE
  '2023-03-10 09:30:00'::TIMESTAMP
  CAST('2023-03-10' AS DATE)

  -- Extraction:
  EXTRACT(YEAR  FROM collected)   -- integer
  EXTRACT(MONTH FROM collected)   -- integer
  DATE_TRUNC('month', collected)  -- truncate to month start: 2023-03-01
  AGE(collected)                  -- interval since collected: '9 mons 21 days'
  TO_DATE('10/03/2023', 'DD/MM/YYYY')  -- parse non-ISO format
""")


=== Date functions on TEXT date columns ===
 result_id  collected parsed_date year month year_month  days_before_year_end
         1 2023-03-10  2023-03-10 2023    03    2023-03                   296
         2 2023-03-15  2023-03-15 2023    03    2023-03                   291
         3 2023-04-01  2023-04-01 2023    04    2023-04                   274
         4 2023-04-12  2023-04-12 2023    04    2023-04                   263
         5 2023-05-05  2023-05-05 2023    05    2023-05                   240
         6 2023-05-20  2023-05-20 2023    05    2023-05                   225
         7 2023-06-01  2023-06-01 2023    06    2023-06                   213
         8 2023-06-18  2023-06-18 2023    06    2023-06                   196
         9 2023-07-02  2023-07-02 2023    07    2023-07                   182
        10 2023-07-15  2023-07-15 2023    07    2023-07                   169

=== PostgreSQL date/time types (reference — not runnable in SQLite) ===

  -- PostgreSQL uses nat

---
## NULL in type expressions


In [5]:
# NULL propagates through almost every operation
print('=== NULL arithmetic and COALESCE ===')
q = """
SELECT
    result_id,
    NULL + 5                          AS null_plus_5,
    NULL || ' suffix'                 AS null_concat,
    COALESCE(NULL, 0)                 AS coalesced_to_zero,
    COALESCE(NULL, NULL, 'fallback')  AS coalesced_chain
FROM lab_results
LIMIT 3
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Safe numeric filter after CAST (CAST of non-numeric text returns NULL in SQLite)
print()
print('=== Filtering after CAST — elevated creatinine > 100 umol/L ===')
q2 = """
SELECT result_id, patient_id, test_name, result_val, units, flagged
FROM   lab_results
WHERE  test_name = 'Creatinine'
  AND  CAST(result_val AS REAL) > 100.0
ORDER BY CAST(result_val AS REAL) DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

# NULLIF — return NULL when two values are equal (useful to avoid divide-by-zero)
print()
print('=== NULLIF to prevent divide-by-zero ===')
q3 = """
SELECT
    10.0 / NULLIF(0, 0)   AS safe_div_by_zero,   -- NULL instead of error
    10.0 / NULLIF(5, 0)   AS safe_div_normal      -- 2.0
"""
print(pd.read_sql(q3, conn).to_string(index=False))

conn.close()


=== NULL arithmetic and COALESCE ===
 result_id null_plus_5 null_concat  coalesced_to_zero coalesced_chain
         1        None        None                  0        fallback
         2        None        None                  0        fallback
         3        None        None                  0        fallback

=== Filtering after CAST — elevated creatinine > 100 umol/L ===
 result_id  patient_id  test_name result_val  units  flagged
        10           9 Creatinine      210.0 umol/L        1
         4           4 Creatinine      145.0 umol/L        1

=== NULLIF to prevent divide-by-zero ===
safe_div_by_zero  safe_div_normal
            None              2.0


---
## Common Pitfalls

**1. Relying on SQLite's silent coercion in code intended for PostgreSQL**
SQLite accepts `INSERT INTO integer_col VALUES ('abc')` — it stores the string. PostgreSQL raises a type error immediately. Always write explicit CASTs and test against a strict-typed database if your code will run on PostgreSQL or another RDBMS.

**2. Integer division silently truncates**
`7 / 2` returns `3` in both SQLite and PostgreSQL when both operands are integers. This silently breaks rate calculations: `COUNT(*) / total` returns 0 for any count smaller than total. Cast at least one operand: `COUNT(*) * 1.0 / total` or `CAST(COUNT(*) AS REAL) / total`.

**3. Comparing TEXT date strings only works correctly with ISO 8601 format**
`WHERE collected > '2023-06-01'` gives correct results only if dates are stored as `YYYY-MM-DD`. The format `DD/MM/YYYY` fails because string comparison would rank `'01/03/2023'` before `'31/01/2023'` — alphabetically but not chronologically. Standardise on ISO 8601 and use explicit `DATE()` functions.

**4. `TYPEOF()` reports storage class, not declared column type**
`TYPEOF(result_val)` returns `'text'` even if you declared the column as `REAL`. SQLite stores whatever was inserted. Use `CAST` explicitly and don't rely on `TYPEOF` to confirm numeric behaviour.

**5. `CAST` of non-numeric text returns NULL in SQLite, error in PostgreSQL**
`CAST('abc' AS REAL)` returns NULL in SQLite with no warning. In PostgreSQL it raises `invalid input syntax for type double precision`. If your data may contain non-numeric values in a numeric column, validate and clean before casting, or use `TRY_CAST` / a `CASE WHEN` guard.

**6. Forgetting NULLIF when computing rates and percentages**
`SUM(value) / COUNT(*)` looks safe but `COUNT(*)` can be zero for an empty group. Use `NULLIF(COUNT(*), 0)` in the denominator to return NULL instead of a division-by-zero error: `SUM(value) / NULLIF(COUNT(*), 0)`.


---
*sql_methods_library - Samantha McGarrigle*